In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.neighbors import KNeighborsClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, recall_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# 文件路径
root_path_DEV = "C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/"
root_path_PROD = "C:/Users/HANJ29/Applications/btweb/stock_filestore/PROD/"
folder_ds_entry = "dataset_entry/"
# folder_ds_tech_funda = "dataset_tech_funda/"
dataset_training_path = root_path_DEV + "dataset_training/"
dataset_training_stat_path = root_path_DEV + "dataset_training/stat/"
dataset_entry_path = root_path_DEV + folder_ds_entry

In [2]:
# 读取CSV文件
def concat_csv_files(folder_path, n=None):
    # 获取文件夹中的所有CSV文件
    csv_files = [
        f
        for f in os.listdir(folder_path)
        if f.endswith(".csv") and f != "timestamp.csv"
    ]

    # 只读取前n个CSV文件
    csv_files = csv_files[0:] if n is None else csv_files[0:n]

    # 初始化一个空的DataFrame列表
    df_list = []

    # 遍历前100个CSV文件并读取到DataFrame中
    for file in csv_files:
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        df_list.append(df)

    # 将所有DataFrame合并为一个
    combined_df = pd.concat(df_list, ignore_index=True)

    # 显示合并后的DataFrame
    # print(combined_df)

    return combined_df


def check_nan_values(df):
    # 统计每列的空值数量
    nan_count = df.isnull().sum()

    # 统计每列的总数
    total_count = df.shape[0]

    # 计算每列空值占总数的比例
    nan_ratio = nan_count / total_count

    # 创建一个 DataFrame 来存储结果
    nan_stats = pd.DataFrame(
        {
            "column_name": nan_count.index,
            "nan_count": nan_count.values,
            "total_count": total_count,
            "nan_ratio": nan_ratio.values,
        }
    )

    print(nan_stats)
    return nan_stats


def drop_nan_values(df, columns_to_drop=["close", "float_mv", "dv_ttm"]):
    # 删除包含 NaN 值的行
    df = df.drop(columns=columns_to_drop)
    df.dropna(
        how="any",
        axis=0,
        inplace=False,
    )
    return df


def drop_columns(df, columns_to_drop=["close", "float_mv", "dv_ttm"]):
    return df.drop(columns=columns_to_drop)

# 预处理数据
def preprocess_data(scan_folder, freq="W", drop_nan=False):
    # tech_funda_path = f"{root_path_DEV}{folder_ds_tech_funda}{freq}/"
    df = concat_csv_files(scan_folder)
    check_nan_values(df)
    return drop_nan_values(df) if drop_nan else df
    # df.to_csv(f"{preprocessed_path}preprocessed_{freq}.csv", index=False)
    # return df


def map_labels(
    df,
    columns=[
        "top_or_bottom",
        "top_or_bottom_stat",
        "top_bottom_volatility_stat",
        "top_or_bottom_stat_optimized",
        "top_or_bottom_optimized",
        "top_bottom_volatility_optimized",
    ],
    class_mapping={"N": 0, "B": 1, "T": 2},
):
    # class_mapping = {"N":0, "B":1, "T":2}
    for column in columns:
        df[column] = df[column].map(class_mapping)
    return df

In [3]:
def check_nan_impact_on_labels(df, columns_to_check, label_columns):
    # 初始化一个字典来存储结果
    results = []

    for column in columns_to_check:
        # 筛选出指定列为空值的行
        nan_rows = df[df[column].isnull()]

        for label_column in label_columns:
            # 统计标签列中每个值的数量
            value_counts = nan_rows[label_column].value_counts()
            for value, count in value_counts.items():
                results.append({
                    'column_with_nan': column,
                    'label_column': label_column,
                    'label_value': value,
                    'count': count
                })

    return pd.DataFrame(results)

In [4]:
# 要检查的列
columns_to_check = [
    "change",
    "pct_chg",
    "vol",
    "atr",
    "pct_vol_chg",
    "pct_o2c",
    "lower_shadow",
    "upper_shadow",
    "dif",
    "dea",
    "bar",
    "rsi_6",
    "rsi_12",
    "rsi_24",
    "k",
    "d",
    "j",
    "turnover_rate",
    "turnover_rate_f",
    "volume_ratio",
    "pe",
    "pe_ttm",  # 当为负值时得到的数据为NaN
    "pb",
    "ps",
    "ps_ttm",
    "dv_ratio",
    # "dv_ttm",
    "total_share",
    "float_share",
    "free_share",
    "total_mv",
    "circ_mv",
    # "float_share_ratio",
    "free_share_ratio",
    "mab_10",
    "mab_25",
    "mab_60",
    "mab_120",
    "mab_200",
]

# 标签列
label_columns = [
    "top_or_bottom",
    "top_or_bottom_stat",
    "top_bottom_volatility_stat",
    "top_or_bottom_stat_optimized",
    "top_or_bottom_optimized",
    "top_bottom_volatility_optimized",
]

拼接个股日线文件成一个文件

In [8]:
freq = "D"
df_d = concat_csv_files(dataset_entry_path + freq + "/")
df_d = map_labels(df_d)
df_d.to_csv(dataset_training_path + "mapped_entry_" + freq + ".csv", index=False)

In [9]:
df_d.head(50)

,ts_code_df1,trade_date,close,open,high,low,pre_close,change,pct_chg,vol,...,top_or_bottom_stat_optimized,top_or_bottom_optimized,top_bottom_volatility_optimized,close_df1,volume_ratio,ps,ps_ttm,dv_ratio,dv_ttm,circ_mv
0,000001.SH,1990-12-19,99.98,96.05,99.98,95.79,100.00,-0.02,-0.0200,1260.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,000001.SH,1990-12-20,104.39,104.30,104.39,99.98,99.98,4.41,4.4109,197.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,000001.SH,1990-12-21,109.13,109.07,109.13,103.73,104.39,4.74,4.5407,28.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,000001.SH,1990-12-24,114.55,113.57,114.55,109.13,109.13,5.42,4.9666,32.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,000001.SH,1990-12-25,120.25,120.09,120.25,114.55,114.55,5.70,4.9760,15.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,000001.SH,1990-12-26,125.27,125.27,125.27,120.25,120.25,5.02,4.1746,100.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,000001.SH,1990-12-27,125.28,125.27,125.28,125.27,125.27,0.01,0.0080,66.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,000001.SH,1990-12-28,126.45,126.39,126.45,125.28,125.28,1.17,0.9339,108.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,000001.SH,1990-12-31,127.61,126.56,127.61,126.48,126.45,1.16,0.9174,78.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,000001.SH,1991-01-02,128.84,127.61,128.84,127.61,127.61,1.23,0.9639,91.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# 统计空值对标签列的影响
freq = "D"
version = "0.2"
nan_val = check_nan_values(df_d)
nan_val.to_csv(f"{dataset_training_path}nan_val_entry_{freq}_{version}.csv", index=False)
print(nan_val)

nan_impact = check_nan_impact_on_labels(df_d, columns_to_check, label_columns)
# 保存结果到 CSV 文件
nan_impact.to_csv(f"{dataset_training_path}nan_impact_{freq}_{version}.csv", index=False)
print(nan_impact)

    column_name  nan_count  total_count  nan_ratio
0   ts_code_df1          0     15561288   0.000000
1    trade_date          0     15561288   0.000000
2         close   15541058     15561288   0.998700
3          open          0     15561288   0.000000
4          high          0     15561288   0.000000
..          ...        ...          ...        ...
61           ps     132325     15561288   0.008503
62       ps_ttm     138588     15561288   0.008906
63     dv_ratio    2267360     15561288   0.145705
64       dv_ttm    6257044     15561288   0.402090
65      circ_mv     102142     15561288   0.006564

[66 rows x 4 columns]
    column_name  nan_count  total_count  nan_ratio
0   ts_code_df1          0     15561288   0.000000
1    trade_date          0     15561288   0.000000
2         close   15541058     15561288   0.998700
3          open          0     15561288   0.000000
4          high          0     15561288   0.000000
..          ...        ...          ...        ...
61      

拼接个股周线文件成一个文件

In [5]:
freq = "W"
df_w = concat_csv_files(dataset_entry_path + freq + "/")
df_w = map_labels(df_w)
df_w.to_csv(dataset_training_path + "mapped_entry_" + freq + ".csv", index=False)
df_w.describe()

,open,high,low,close_df1,pre_close,change,pct_chg,vol,amount,atr,...,total_mv,circ_mv,float_share_ratio,free_share_ratio,top_or_bottom,top_or_bottom_stat,top_bottom_volatility_stat,top_or_bottom_stat_optimized,top_or_bottom_optimized,top_bottom_volatility_optimized
count,3.279959e+06,3.279959e+06,3.279959e+06,3.279959e+06,3.279959e+06,3.279959e+06,3.279959e+06,3.279959e+06,3.279959e+06,3.204219e+06,...,3.279925e+06,3.279929e+06,3.279925e+06,3.274802e+06,3.279959e+06,3.279959e+06,3.279959e+06,3.279959e+06,3.279959e+06,3.279959e+06
mean,1.319641e+01,1.390828e+01,1.255188e+01,1.317775e+01,1.725483e+01,-4.077078e+00,-1.855142e+01,5.835930e+07,7.067795e+08,1.358769e+00,...,1.502222e+06,1.027798e+06,6.751812e-01,4.451422e-01,6.370394e-01,1.334907e-01,8.585565e-02,1.148877e-01,5.012532e-01,7.501069e-02
std,2.523806e+01,2.644688e+01,2.410867e+01,2.521521e+01,3.082707e+01,1.388746e+01,2.648386e+01,1.546919e+08,1.826611e+09,2.482925e+00,...,7.293764e+06,4.967089e+06,2.941935e-01,1.866290e-01,8.088448e-01,4.511917e-01,3.665592e-01,4.214897e-01,7.643575e-01,3.442533e-01
min,8.880000e-02,9.010000e-02,8.880000e-02,8.920000e-02,1.600000e-01,-2.877528e+03,-9.997310e+01,1.000000e+01,2.880000e+02,1.000000e-02,...,1.184780e+03,1.184780e+03,9.948656e-03,4.060404e-04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,4.897300e+00,5.121100e+00,4.690000e+00,4.896700e+00,6.560000e+00,-4.198700e+00,-3.508055e+01,7.503059e+06,8.883277e+07,4.200000e-01,...,2.477239e+05,1.258046e+05,4.010018e-01,3.000000e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,8.120000e+00,8.512300e+00,7.762900e+00,8.118600e+00,1.088000e+01,-6.781000e-01,-6.459900e+00,2.101569e+07,2.460234e+08,7.600000e-01,...,4.478530e+05,2.946522e+05,7.176213e-01,4.355930e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.424175e+01,1.498720e+01,1.357060e+01,1.422335e+01,1.870000e+01,-1.910000e-02,-2.314000e-01,5.559488e+07,6.538328e+08,1.430000e+00,...,9.507919e+05,6.790959e+05,9.940941e-01,5.829548e-01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00
max,2.563742e+03,2.603268e+03,2.359294e+03,2.576640e+03,2.880000e+03,3.652800e+02,1.214754e+03,1.971536e+10,2.608253e+11,1.564000e+02,...,7.104874e+08,3.267370e+08,1.003501e+00,1.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00


In [8]:
# 统计每个 ts_code 的空值数量
missing_counts = df_w['turnover_rate'].isna().groupby(df_w['ts_code_df1']).sum()

# 打印结果
print("每个 ts_code 的 turnover_rate 空值数量：")
print(missing_counts)
# 保存统计结果为 CSV 文件
missing_counts.to_csv(
    f"{dataset_training_stat_path}to_nan_value_counts_{freq}.csv", index_label="value"
)

每个 ts_code 的 turnover_rate 空值数量：
ts_code_df1
000001.SZ     0
000002.SZ     0
000004.SZ    57
000005.SZ     9
000006.SZ    57
             ..
873703.BJ     0
873706.BJ     0
873726.BJ     0
873806.BJ     0
873833.BJ     0
Name: turnover_rate, Length: 5406, dtype: int64


In [6]:
# 统计空值对标签列的影响
# freq = "W"
version = "0.3"
nan_val = check_nan_values(df_w)
nan_val.to_csv(f"{dataset_training_path}nan_val_entry_{freq}_{version}.csv", index=False)
print(nan_val)

nan_impact = check_nan_impact_on_labels(df_w, columns_to_check, label_columns)
# 保存结果到 CSV 文件
nan_impact.to_csv(f"{dataset_training_path}nan_impact_{freq}_{version}.csv", index=False)
print(nan_impact)

                        column_name  nan_count  total_count  nan_ratio
0                       ts_code_df1          0      3274453        0.0
1                        trade_date          0      3274453        0.0
2                              open          0      3274453        0.0
3                              high          0      3274453        0.0
4                               low          0      3274453        0.0
..                              ...        ...          ...        ...
59               top_or_bottom_stat          0      3274453        0.0
60       top_bottom_volatility_stat          0      3274453        0.0
61     top_or_bottom_stat_optimized          0      3274453        0.0
62          top_or_bottom_optimized          0      3274453        0.0
63  top_bottom_volatility_optimized          0      3274453        0.0

[64 rows x 4 columns]
                        column_name  nan_count  total_count  nan_ratio
0                       ts_code_df1          0      32

拼接个股月线文件成一个文件

In [13]:
freq = "M"
df_m = concat_csv_files(dataset_entry_path + freq + "/")
df_m = map_labels(df_m)
df_m.to_csv(dataset_training_path + "mapped_entry_" + freq + ".csv", index=False)
df_m.describe()

,open,high,low,close_df1,pre_close,change,pct_chg,vol,amount,atr,...,total_mv,circ_mv,float_share_ratio,free_share_ratio,top_or_bottom,top_or_bottom_stat,top_bottom_volatility_stat,top_or_bottom_stat_optimized,top_or_bottom_optimized,top_bottom_volatility_optimized
count,778853.000000,778853.000000,778853.000000,778853.000000,778853.00000,778853.000000,778853.000000,7.788530e+05,7.788530e+05,703187.000000,...,7.788460e+05,7.788460e+05,778845.000000,777670.000000,778853.000000,778853.000000,778853.000000,778853.000000,778853.000000,778853.000000
mean,12.842080,14.413994,11.473223,12.745099,12.77477,-0.029671,1.478154,2.430937e+08,2.944557e+09,2.789383,...,1.484688e+06,1.013507e+06,0.672149,0.444233,0.665445,0.147127,0.091916,0.131013,0.533009,0.083152
std,24.015946,26.466184,21.654030,23.844332,23.86031,4.249480,24.066825,6.104519e+08,7.250877e+09,4.458520,...,7.218987e+06,4.915489e+06,0.294672,0.186502,0.817249,0.468781,0.376696,0.444901,0.776445,0.359348
min,0.090000,0.090000,0.090000,0.090000,0.09000,-484.600000,-95.730000,1.000000e+01,2.440000e+02,0.080000,...,1.196638e+03,1.196638e+03,0.009949,0.000406,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.830000,5.360000,4.400000,4.820000,4.81000,-0.600000,-7.420000,3.433458e+07,4.144539e+08,1.050000,...,2.449415e+05,1.228945e+05,0.398471,0.299194,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,8.010000,8.930000,7.260000,7.990000,7.99000,-0.020000,-0.410000,9.289316e+07,1.106200e+09,1.750000,...,4.431564e+05,2.899522e+05,0.711632,0.434418,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,14.000000,15.680000,12.570000,13.930000,13.96000,0.540000,7.450000,2.387698e+08,2.819105e+09,3.030000,...,9.393927e+05,6.697229e+05,0.993352,0.581809,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,2222.000000,2603.270000,2047.940000,2197.230000,2218.00000,300.120000,3484.850000,4.587650e+10,6.611574e+11,240.520000,...,5.768821e+08,2.786247e+08,1.003501,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000


In [14]:
# 统计空值对标签列的影响
freq = "M"
version = "0.3"
nan_val = check_nan_values(df_m)
nan_val.to_csv(f"{dataset_training_path}nan_val_entry_{freq}_{version}.csv", index=False)
print(nan_val)

nan_impact = check_nan_impact_on_labels(df_m, columns_to_check, label_columns)
# 保存结果到 CSV 文件
nan_impact.to_csv(f"{dataset_training_path}nan_impact_{freq}_{version}.csv", index=False)
print(nan_impact)

                        column_name  nan_count  total_count  nan_ratio
0                       ts_code_df1          0       778853        0.0
1                        trade_date          0       778853        0.0
2                              open          0       778853        0.0
3                              high          0       778853        0.0
4                               low          0       778853        0.0
..                              ...        ...          ...        ...
59               top_or_bottom_stat          0       778853        0.0
60       top_bottom_volatility_stat          0       778853        0.0
61     top_or_bottom_stat_optimized          0       778853        0.0
62          top_or_bottom_optimized          0       778853        0.0
63  top_bottom_volatility_optimized          0       778853        0.0

[64 rows x 4 columns]
                        column_name  nan_count  total_count  nan_ratio
0                       ts_code_df1          0       7

freq = "D"
df_d = pd.read_csv(dataset_training_path + "raw_entry_" + freq + ".csv",)

In [11]:
col = ["close", "float_mv", "dv_ttm"]
df_d = drop_columns(df_d, columns_to_drop=col)
df_d = map_labels(df_d)

In [12]:
df_d.describe()

c:\Users\HANJ29\Applications\venv-stock-insider-admin\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\HANJ29\Applications\venv-stock-insider-admin\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\HANJ29\Applications\venv-stock-insider-admin\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,open,high,low,pre_close,change,pct_chg,vol,amount,atr,sl_atr,...,top_bottom_volatility_stat,top_or_bottom_stat_optimized,top_or_bottom_optimized,top_bottom_volatility_optimized,close_df1,volume_ratio,ps,ps_ttm,dv_ratio,circ_mv
count,1.554402e+07,1.554402e+07,1.554402e+07,1.554401e+07,1.554401e+07,15544014.00,1.554402e+07,1.554380e+07,1.546830e+07,1.546830e+07,...,1.554402e+07,1.554402e+07,1.554402e+07,1.554402e+07,1.552380e+07,1.539421e+07,1.539147e+07,1.538521e+07,1.325487e+07,1.542190e+07
mean,1.821964e+01,1.856925e+01,1.789986e+01,1.823021e+01,5.207101e-03,inf,2.644127e+05,3.264306e+05,6.951874e-01,1.753596e+01,...,7.936006e-02,1.079904e-01,5.057373e-01,6.932958e-02,1.284888e+01,1.101002e+00,5.075111e+01,3.733824e+04,1.049795e+00,1.024612e+06
std,2.051450e+02,2.072796e+02,2.029215e+02,2.052477e+02,3.819844e+00,NaN,6.630951e+06,8.410068e+06,5.064395e+00,2.011807e+02,...,3.545713e-01,4.108483e-01,7.662571e-01,3.324967e-01,2.397571e+01,6.211434e+00,6.615038e+03,1.362626e+07,1.824367e+00,4.966090e+06
min,9.000000e-02,9.000000e-02,9.000000e-02,0.000000e+00,-1.293659e+03,-96.61,0.000000e+00,0.000000e+00,0.000000e+00,-9.700000e-01,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9.000000e-02,0.000000e+00,1.060000e-02,1.060000e-02,0.000000e+00,1.107705e+03
25%,4.860000e+00,4.960000e+00,4.780000e+00,4.870000e+00,-1.100000e-01,-1.47,1.492895e+04,1.758356e+04,1.700000e-01,4.670000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.860000e+00,6.900000e-01,1.638000e+00,1.578000e+00,0.000000e+00,1.239660e+05
50%,8.050000e+00,8.220000e+00,7.900000e+00,8.060000e+00,0.000000e+00,0.00,4.258206e+04,4.943218e+04,3.100000e-01,7.720000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,8.050000e+00,9.100000e-01,3.399000e+00,3.254300e+00,4.589000e-01,2.921338e+05
75%,1.405000e+01,1.436000e+01,1.378000e+01,1.406000e+01,1.100000e-01,1.45,1.146210e+05,1.346889e+05,6.100000e-01,1.343000e+01,...,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,1.403000e+01,1.230000e+00,7.022800e+00,6.674700e+00,1.316400e+00,6.760354e+05
max,1.955458e+04,1.960003e+04,1.920311e+04,1.953115e+04,1.254795e+03,inf,1.587912e+09,1.941426e+09,9.335800e+02,1.901848e+04,...,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.576640e+03,1.845921e+04,2.223993e+06,6.318763e+09,6.780270e+01,3.267370e+08


In [8]:
freq = "D"
df_d = pd.read_csv(dataset_training_path + "mapped_entry_" + freq + ".csv",)

In [28]:
df_d.to_csv(dataset_training_path + "mapped_entry_" + freq + ".csv", index=False)

In [22]:
df_d.columns

Index(['ts_code_df1', 'trade_date', 'close', 'open', 'high', 'low',
       'pre_close', 'change', 'pct_chg', 'vol', 'amount', 'atr', 'sl_atr',
       'sl_2atr', 'sl_3atr', 'sl_4atr', 'sl_5atr', 'pct_vol_chg',
       'pct_amount_chg', 'close_atr_diff', 'close_2atr_diff',
       'close_3atr_diff', 'close_4atr_diff', 'close_5atr_diff', 'pct_o2c',
       'lower_shadow', 'upper_shadow', 'dif', 'dea', 'bar', 'rsi_6', 'rsi_12',
       'rsi_24', 'k', 'd', 'j', 'total_mv', 'float_mv', 'total_share',
       'float_share', 'free_share', 'turnover_rate', 'turnover_rate_f', 'pe',
       'pe_ttm', 'pb', 'float_share_ratio', 'free_share_ratio',
       'top_or_bottom', 'top_or_bottom_stat', 'top_bottom_volatility_stat',
       'top_or_bottom_stat_optimized', 'top_or_bottom_optimized',
       'top_bottom_volatility_optimized', 'close_df1', 'volume_ratio', 'ps',
       'ps_ttm', 'dv_ratio', 'dv_ttm', 'circ_mv'],
      dtype='object')

In [24]:
df_w.describe()

,open,high,low,close_df1,pre_close,change,pct_chg,vol,amount,atr,...,ps_ttm,dv_ratio,dv_ttm,total_share,float_share,free_share,total_mv,circ_mv,float_share_ratio,free_share_ratio
count,3.274190e+06,3.274190e+06,3.274190e+06,3.274190e+06,3.274190e+06,3.274190e+06,3.274190e+06,3.274190e+06,3.274190e+06,3.198534e+06,...,3.086295e+06,2.659547e+06,1.872023e+06,3.093601e+06,3.093603e+06,3.088530e+06,3.093599e+06,3.093603e+06,3.093599e+06,3.088530e+06
mean,1.285580e+01,1.355287e+01,1.222768e+01,1.284066e+01,1.284301e+01,-2.345954e-03,3.604999e-01,5.820295e+07,7.054825e+08,1.325882e+00,...,3.795818e+04,1.049396e+00,1.809902e+00,1.513302e+05,1.037508e+05,4.816996e+04,1.493761e+06,1.019860e+06,6.711996e-01,4.432719e-01
std,2.401288e+01,2.514966e+01,2.295316e+01,2.399577e+01,2.399368e+01,2.117932e+00,1.095291e+01,1.547760e+08,1.827552e+09,2.328748e+00,...,1.377782e+07,1.829494e+00,1.049851e+01,1.111990e+06,7.492612e+05,1.452185e+05,7.281338e+06,4.951269e+06,2.951041e-01,1.865554e-01
min,9.000000e-02,9.000000e-02,9.000000e-02,9.000000e-02,9.000000e-02,-3.340600e+02,-9.876000e+01,3.000000e+00,2.440000e+02,1.000000e-02,...,1.070000e-02,0.000000e+00,0.000000e+00,3.293800e+00,3.293800e+00,3.293800e+00,1.107705e+03,1.107705e+03,9.948656e-03,4.060404e-04
25%,4.870000e+00,5.090000e+00,4.660000e+00,4.870000e+00,4.870000e+00,-2.700000e-01,-3.360000e+00,7.475681e+06,8.843373e+07,4.200000e-01,...,1.577200e+00,0.000000e+00,4.598000e-01,1.982438e+04,9.056200e+03,7.101600e+03,2.461389e+05,1.231801e+05,3.966791e-01,2.979269e-01
50%,8.050000e+00,8.440000e+00,7.700000e+00,8.050000e+00,8.050000e+00,0.000000e+00,0.000000e+00,2.095159e+07,2.452139e+08,7.500000e-01,...,3.254700e+00,4.563000e-01,9.726000e-01,4.048177e+04,2.662965e+04,1.737184e+04,4.456824e+05,2.906417e+05,7.100119e-01,4.332478e-01
75%,1.403000e+01,1.477000e+01,1.337000e+01,1.402000e+01,1.402000e+01,2.500000e-01,3.340000e+00,5.540631e+07,6.522117e+08,1.420000e+00,...,6.681300e+00,1.314900e+00,1.967700e+00,8.729410e+04,6.824869e+04,4.222276e+04,9.463435e+05,6.725621e+05,9.933248e-01,5.808156e-01
max,2.563740e+03,2.603270e+03,2.359290e+03,2.576640e+03,2.576640e+03,3.596600e+02,3.800000e+03,1.971536e+10,2.608253e+11,1.564000e+02,...,6.313728e+09,6.688640e+01,1.787040e+03,3.564063e+07,3.192442e+07,5.564158e+06,7.104874e+08,3.090247e+08,1.003501e+00,1.000000e+00


In [ ]:
df_w.read_csv(dataset_training_path + "raw_entry_" + freq + ".csv",)

In [11]:
freq = "W"
df_w = pd.read_csv(dataset_training_path + "mapped_entry_" + freq + ".csv",)

In [12]:
# 统计空值对标签列的影响
nan_impact = check_nan_impact_on_labels(df_w, columns_to_check, label_columns)
print(nan_impact)

      column_with_nan                     label_column  label_value   count
0                 atr                    top_or_bottom            0   46270
1                 atr                    top_or_bottom            1   14938
2                 atr                    top_or_bottom            2   14448
3                 atr               top_or_bottom_stat            0   69522
4                 atr               top_or_bottom_stat            2    4298
..                ...                              ...          ...     ...
451  free_share_ratio          top_or_bottom_optimized            1   37586
452  free_share_ratio          top_or_bottom_optimized            2   22150
453  free_share_ratio  top_bottom_volatility_optimized            0  172893
454  free_share_ratio  top_bottom_volatility_optimized            1    9505
455  free_share_ratio  top_bottom_volatility_optimized            2    3262

[456 rows x 4 columns]


In [13]:
nan_impact.to_csv(f"{dataset_training_path}nan_impact_{freq}.csv", index=False)

In [25]:
nan_val = check_nan_values(df_w)
nan_val.to_csv(dataset_training_path + "nan_val_entry_" + freq + ".csv", index=False)

                        column_name  nan_count  total_count  nan_ratio
0                       ts_code_df1          0      3274190   0.000000
1                        trade_date          0      3274190   0.000000
2                              open          0      3274190   0.000000
3                              high          0      3274190   0.000000
4                               low          0      3274190   0.000000
5                         close_df1          0      3274190   0.000000
6                         pre_close          0      3274190   0.000000
7                            change          0      3274190   0.000000
8                           pct_chg          0      3274190   0.000000
9                               vol          0      3274190   0.000000
10                           amount          0      3274190   0.000000
11                              atr      75656      3274190   0.023107
12                           sl_atr      75656      3274190   0.023107
13    

In [26]:
def clean_raw_data(df, freq="D", drop_col=False, col=["close", "float_mv", "dv_ttm"]):
    if drop_col:
        df = drop_columns(df, columns_to_drop=col)
    df = map_labels(df)
    # 统计空值对标签列的影响
    nan_impact = check_nan_impact_on_labels(df, columns_to_check, label_columns)
    nan_impact.to_csv(f"{dataset_training_path}nan_impact_{freq}.csv", index=False)
    print(nan_impact)

In [27]:
col = ["close", "float_mv", "dv_ttm"]
clean_raw_data(df_w, freq="W", drop_col=False)

   column_with_nan                     label_column  label_value   count
0               pe                    top_or_bottom            0  306704
1               pe                    top_or_bottom            1  111577
2               pe                    top_or_bottom            2   98999
3               pe               top_or_bottom_stat            0  468381
4               pe               top_or_bottom_stat            1   29447
5               pe               top_or_bottom_stat            2   19452
6               pe       top_bottom_volatility_stat            0  483756
7               pe       top_bottom_volatility_stat            1   20669
8               pe       top_bottom_volatility_stat            2   12855
9               pe     top_or_bottom_stat_optimized            0  474049
10              pe     top_or_bottom_stat_optimized            1   26267
11              pe     top_or_bottom_stat_optimized            2   16964
12              pe          top_or_bottom_optimized

In [29]:
df_w

,ts_code_df1,trade_date,open,high,low,close_df1,pre_close,change,pct_chg,vol,...,total_mv,circ_mv,float_share_ratio,free_share_ratio,top_or_bottom,top_or_bottom_stat,top_bottom_volatility_stat,top_or_bottom_stat_optimized,top_or_bottom_optimized,top_bottom_volatility_optimized
0,000001.SZ,1991-07-05,1.95,1.95,1.91,1.91,1.96,-0.05,-2.55,1700.0,...,159905.0638,87370.5000,0.546390,0.546390,0,0,0,0,0,0
1,000001.SZ,1991-07-12,1.89,1.89,1.86,1.86,1.90,-0.04,-2.11,1600.0,...,156025.0501,85250.5000,0.546390,0.546390,0,0,0,0,0,0
2,000001.SZ,1991-07-19,1.83,1.83,1.80,1.80,1.84,-0.04,-2.17,500.0,...,150593.0310,82282.5000,0.546390,0.546390,0,0,0,0,0,0
3,000001.SZ,1991-07-26,1.76,1.76,1.74,1.74,1.77,-0.03,-1.69,500.0,...,146179.5154,79871.0000,0.546390,0.546390,0,0,0,0,0,0
4,000001.SZ,1991-08-02,1.73,1.73,1.69,1.69,1.73,-0.04,-2.31,2300.0,...,141814.5000,77486.0000,0.546390,0.546390,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3274185,873833.BJ,2025-02-14,12.50,13.66,12.30,13.16,12.53,0.63,5.03,15972595.0,...,108385.7600,41753.1715,0.385227,0.317074,2,0,0,0,2,0
3274186,873833.BJ,2025-02-21,13.17,13.40,11.87,12.86,13.16,-0.30,-2.28,15230398.0,...,105914.9600,40801.3515,0.385227,0.326661,1,0,0,0,1,0
3274187,873833.BJ,2025-02-28,12.96,14.59,12.71,13.54,12.86,0.68,5.29,18725586.0,...,111515.4400,42958.8102,0.385227,0.328795,0,0,0,0,0,0
3274188,873833.BJ,2025-03-07,13.42,17.29,13.30,15.56,13.54,2.02,14.92,32338574.0,...,137294.1200,52889.4658,0.385227,0.328795,0,0,0,0,0,0


In [31]:
df_w.to_csv(dataset_training_path + "mapped_entry_" + freq + ".csv", index=False)

In [14]:
freq = "M"
df_m = pd.read_csv(dataset_training_path + "mapped_entry_" + freq + ".csv",)

In [15]:
# 统计空值对标签列的影响
nan_impact = check_nan_impact_on_labels(df_m, columns_to_check, label_columns)
print(nan_impact)

      column_with_nan                     label_column  label_value   count
0                 atr                    top_or_bottom            0   43509
1                 atr                    top_or_bottom            1   15795
2                 atr                    top_or_bottom            2   15616
3                 atr               top_or_bottom_stat            0   68479
4                 atr               top_or_bottom_stat            2    3585
..                ...                              ...          ...     ...
451  free_share_ratio          top_or_bottom_optimized            1   55055
452  free_share_ratio          top_or_bottom_optimized            2   39940
453  free_share_ratio  top_bottom_volatility_optimized            0  246740
454  free_share_ratio  top_bottom_volatility_optimized            1   10390
455  free_share_ratio  top_bottom_volatility_optimized            2    5572

[456 rows x 4 columns]


In [16]:
nan_impact.to_csv(f"{dataset_training_path}nan_impact_{freq}.csv", index=False)

In [33]:
nan_val = check_nan_values(df_m)
nan_val.to_csv(dataset_training_path + "nan_val_entry_" + freq + ".csv", index=False)

                        column_name  nan_count  total_count  nan_ratio
0                       ts_code_df1          0       775685   0.000000
1                        trade_date          0       775685   0.000000
2                              open          0       775685   0.000000
3                              high          0       775685   0.000000
4                               low          0       775685   0.000000
5                         close_df1          0       775685   0.000000
6                         pre_close          0       775685   0.000000
7                            change          0       775685   0.000000
8                           pct_chg          0       775685   0.000000
9                               vol          0       775685   0.000000
10                           amount          0       775685   0.000000
11                              atr      74920       775685   0.096586
12                           sl_atr      74920       775685   0.096586
13    

In [34]:
col = ["close", "float_mv", "dv_ttm"]
clean_raw_data(df_m, freq="M", drop_col=False)

   column_with_nan                     label_column  label_value   count
0               pe                    top_or_bottom            0  176792
1               pe                    top_or_bottom            1   76674
2               pe                    top_or_bottom            2   63398
3               pe               top_or_bottom_stat            0  283822
4               pe               top_or_bottom_stat            1   19986
5               pe               top_or_bottom_stat            2   13056
6               pe       top_bottom_volatility_stat            0  296072
7               pe       top_bottom_volatility_stat            1   12907
8               pe       top_bottom_volatility_stat            2    7885
9               pe     top_or_bottom_stat_optimized            0  287219
10              pe     top_or_bottom_stat_optimized            1   18081
11              pe     top_or_bottom_stat_optimized            2   11564
12              pe          top_or_bottom_optimized

In [39]:
df_m.to_csv(dataset_training_path + "mapped_entry_" + freq + ".csv", index=False)

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split


def split_and_save_dataset_by_type(file_path, freq="D", test_size=0.2, random_state=42):
    # 读取数据集
    df = pd.read_csv(file_path + f"preprocessed_mapped_FULL_{freq}.csv")
    df = df.rename(columns={"ts_code_df1": "ts_code"})
    
    # 保存完整的不经过过滤的分割数据
    train_df, test_df = train_test_split(df, test_size=test_size, random_state=random_state)
    train_df.to_csv(file_path + f"train_dataset_full_{freq}.csv", index=False)
    test_df.to_csv(file_path + f"test_dataset_full_{freq}.csv", index=False)
    print("完整数据集已分割并保存为 train_dataset_full.csv 和 test_dataset_full.csv")

    # 根据 ts_code 字段将数据集分为不同类型的数据
    types = {
        "GEM": df[
            df["ts_code"].str.startswith("300") | df["ts_code"].str.startswith("301") #创业板
        ],
        "STAR": df[df["ts_code"].str.startswith("688")], #科创板
        "SME": df[df["ts_code"].str.startswith("002")], #中小板
        "SZMAIN": df[
            df["ts_code"].str.startswith("000") | df["ts_code"].str.startswith("001")
        ], #深圳主板
        "SHMAIN": df[
            df["ts_code"].str.startswith("600")
            | df["ts_code"].str.startswith("601")
            | df["ts_code"].str.startswith("603")
        ], #上海主板
    }

    for type_name, type_df in types.items():
        # 按照 80% 的训练和 20% 的测试比例分割数据
        train_df, test_df = train_test_split(
            type_df, test_size=test_size, random_state=random_state
        )

        # 保存分割后的数据集为 CSV 文件
        train_file_name = file_path + f"train_dataset_{type_name}_{freq}.csv"
        test_file_name = file_path + f"test_dataset_{type_name}_{freq}.csv"
        train_df.to_csv(train_file_name, index=False)
        test_df.to_csv(test_file_name, index=False)

        print(f"数据集已分割并保存为 {train_file_name} 和 {test_file_name}")
        
def split_and_save_dataset_by_market(file_path, freq="D", test_size=0.2, random_state=42):
    # 读取数据集
    df = pd.read_csv(file_path + f"preprocessed_mapped_FULL_{freq}.csv")
    df = df.rename(columns={"ts_code_df1": "ts_code"})

    # 根据 ts_code 字段将数据集分为不同类型的数据
    types = {
        "GEM": df[df["ts_code"].str.startswith("300") | df["ts_code"].str.startswith("301")],  # 创业板
        "STAR": df[df["ts_code"].str.startswith("688")],  # 科创板
        "SME": df[df["ts_code"].str.startswith("002")],  # 中小板
        "SZMAIN": df[df["ts_code"].str.startswith("000") | df["ts_code"].str.startswith("001")],  # 深圳主板
        "SHMAIN": df[df["ts_code"].str.startswith("600") | df["ts_code"].str.startswith("601") | df["ts_code"].str.startswith("603")],  # 上海主板
    }

    for type_name, type_df in types.items():
        # 保存分类后的数据集为 CSV 文件
        type_file_name = file_path + f"{type_name}_dataset_{freq}.csv"
        type_df.to_csv(type_file_name, index=False)
        print(f"数据集已分类并保存为 {type_file_name}")

In [8]:
df_m = pd.read_csv(dataset_training_path + "/preprocessed_mapped_FULL_M.csv")
df_m.head()

,ts_code_df1,trade_date,change,pct_chg,vol,atr,pct_vol_chg,pct_o2c,lower_shadow,upper_shadow,...,dv_ratio_nearest,dv_ttm_nearest,float_share_ratio_nearest,free_share_ratio_nearest,top_or_bottom,top_or_bottom_stat,top_bottom_volatility_stat,top_or_bottom_stat_optimized,top_or_bottom_optimized,top_bottom_volatility_optimized
0,000001.SZ,1991-07-31,-32.1113,-94.9477,5800.0,NaN,NaN,-0.139,0.000,0.000,...,0.1,0.1,0.1,0.1,0,0.0,0.0,0,0,0
1,000001.SZ,1991-08-30,-27.9287,-94.5454,2757300.0,NaN,0.998,-0.953,0.052,0.000,...,0.1,0.1,0.1,0.1,2,0.0,0.0,0,2,0
2,000001.SZ,1991-09-30,-13.4476,-89.6507,6213700.0,NaN,0.556,-0.035,0.074,0.000,...,0.1,0.1,0.1,0.1,1,NaN,NaN,1,1,1
3,000001.SZ,1991-10-31,-11.7199,-80.2733,9587900.0,NaN,0.352,0.455,0.000,0.035,...,0.5,0.5,0.5,0.5,0,0.0,0.0,0,0,0
4,000001.SZ,1991-11-29,-23.9857,-89.1662,10683600.0,NaN,0.103,0.031,0.000,0.336,...,0.5,0.5,0.5,0.5,2,0.0,0.0,0,2,0


In [8]:
split_and_save_dataset_by_market(dataset_training_path, freq="M", test_size=0.2, random_state=42)

数据集已分类并保存为 C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/GEM_dataset_M.csv
数据集已分类并保存为 C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/STAR_dataset_M.csv
数据集已分类并保存为 C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/SME_dataset_M.csv
数据集已分类并保存为 C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/SZMAIN_dataset_M.csv
数据集已分类并保存为 C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/SHMAIN_dataset_M.csv


In [ ]:
pre_df = preprocess_data(freq="D")
pre_df.head()

In [ ]:
pre_df_d = map_labels(pre_df)
pre_df_d.to_csv(f"{dataset_training_path}preprocessed_mapped_D.csv", index=False)

In [ ]:
def preprocess_bundle(freq="D"):
    pre_df = preprocess_data(freq=freq)
    pre_df = map_labels(pre_df)
    pre_df.to_csv(f"{dataset_training_path}preprocessed_mapped_{freq}.csv", index=False)
    return pre_df

In [15]:
pre_df_w = preprocess_bundle(freq="W")

NaN counts per column:
ts_code_df1                        False
trade_date                         False
change                             False
pct_chg                            False
vol                                False
                                   ...  
top_or_bottom_stat                 False
top_bottom_volatility_stat         False
top_or_bottom_stat_optimized       False
top_or_bottom_optimized            False
top_bottom_volatility_optimized    False
Length: 70, dtype: bool


In [16]:
pre_df_m = preprocess_bundle(freq="M")

NaN counts per column:
ts_code_df1                        False
trade_date                         False
change                             False
pct_chg                            False
vol                                False
                                   ...  
top_or_bottom_stat                 False
top_bottom_volatility_stat         False
top_or_bottom_stat_optimized       False
top_or_bottom_optimized            False
top_bottom_volatility_optimized    False
Length: 70, dtype: bool


In [17]:
pre_df_m.columns

Index(['ts_code_df1', 'trade_date', 'change', 'pct_chg', 'vol', 'atr',
       'pct_vol_chg', 'pct_o2c', 'lower_shadow', 'upper_shadow', 'dif', 'dea',
       'bar', 'rsi_6', 'rsi_12', 'rsi_24', 'k', 'd', 'j', 'turnover_rate',
       'turnover_rate_f', 'volume_ratio', 'pe', 'pe_ttm', 'pb', 'ps', 'ps_ttm',
       'dv_ratio', 'dv_ttm', 'total_share', 'float_share', 'free_share',
       'total_mv', 'circ_mv', 'float_share_ratio', 'free_share_ratio',
       'change_nearest', 'pct_chg_nearest', 'pct_vol_chg_nearest',
       'vol_nearest', 'k_nearest', 'd_nearest', 'j_nearest', 'dif_nearest',
       'dea_nearest', 'bar_nearest', 'rsi_6_nearest', 'rsi_12_nearest',
       'rsi_24_nearest', 'atr_nearest', 'turnover_rate_nearest',
       'turnover_rate_f_nearest', 'volume_ratio_nearest', 'pct_o2c_nearest',
       'upper_shadow_nearest', 'pe_nearest', 'pe_ttm_nearest', 'pb_nearest',
       'ps_nearest', 'ps_ttm_nearest', 'dv_ratio_nearest', 'dv_ttm_nearest',
       'float_share_ratio_nearest', 'fre

In [12]:
freq = "D"
tscode = "000002.SZ"
tech_funda_fin = pd.read_csv(f"{root_path_DEV}dataset_tech_funda_fin/{freq}/" + tscode + ".csv")
tech_funda_fin.head()

,trade_date,ts_code_df1,open,high,low,close_df1,pre_close,change,pct_chg,vol,...,withdra_biz_devfund,withdra_rese_fund,withdra_oth_ersu,workers_welfare,distr_profit_shrhder,prfshare_payable_dvd,comshare_payable_dvd,capit_comstock_div,continued_net_profit,update_flag
0,1991-01-29,000002.SZ,1.60,1.60,1.60,1.60,0.11,1.49,1354.55,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.179780e+10,0.0
1,1991-01-30,000002.SZ,1.60,1.60,1.60,1.60,1.60,0.00,0.00,17.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.179780e+10,0.0
2,1991-02-04,000002.SZ,1.60,1.60,1.60,1.60,1.60,0.00,0.00,56.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.179780e+10,0.0
3,1991-02-05,000002.SZ,1.61,1.61,1.61,1.61,1.60,0.01,0.63,29.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.179780e+10,0.0
4,1991-02-06,000002.SZ,1.62,1.62,1.62,1.62,1.61,0.01,0.62,29.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.179780e+10,0.0


In [13]:
tech_funda_fin.describe()

,open,high,low,close_df1,pre_close,change,pct_chg,vol,amount,atr,...,withdra_biz_devfund,withdra_rese_fund,withdra_oth_ersu,workers_welfare,distr_profit_shrhder,prfshare_payable_dvd,comshare_payable_dvd,capit_comstock_div,continued_net_profit,update_flag
count,8078.000000,8078.000000,8078.000000,8078.000000,8078.000000,8078.000000,8078.000000,8.078000e+03,8.078000e+03,8078.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.078000e+03,8078.000000
mean,9.642910,9.830993,9.477804,9.653310,9.649312,0.003999,0.270420,5.848796e+05,8.547432e+05,0.372264,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.045922e+10,0.188165
std,7.836288,7.974892,7.717855,7.845345,7.844522,0.325636,15.359391,8.335107e+05,1.240053e+06,0.308414,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.279409e+09,0.390868
min,0.760000,0.760000,0.750000,0.760000,0.110000,-2.540000,-28.360000,1.000000e+00,6.000000e+00,0.010000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.639811e+10,0.000000
25%,2.810000,2.890000,2.720000,2.802500,2.800000,-0.100000,-1.360000,2.203450e+04,2.692696e+04,0.140000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.179780e+10,0.000000
50%,7.435000,7.575000,7.330000,7.440000,7.440000,0.000000,0.000000,3.538948e+05,3.678269e+05,0.300000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.179780e+10,0.000000
75%,13.560000,13.785000,13.320000,13.530000,13.530000,0.080000,1.240000,7.979782e+05,1.250262e+06,0.530000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.179780e+10,0.000000
max,36.680000,37.680000,36.100000,36.690000,36.690000,2.770000,1354.550000,1.097234e+07,2.010649e+07,1.780000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.929812e+10,1.000000


In [23]:
col_to_cal_pe = ["trade_date", "pe", "pe_ttm", "n_income", "eps", "basic_eps", "diluted_eps", "total_revenue", "revenue", "total_share_x", "total_share_y", "close_df1"]
tech_funda_fin_filtered = tech_funda_fin[col_to_cal_pe]
tech_funda_fin_filtered.head(50)

,trade_date,pe,pe_ttm,n_income,eps,basic_eps,diluted_eps,total_revenue,revenue,total_share_x,total_share_y,close_df1
0,1991-01-29,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.60
1,1991-01-30,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.60
2,1991-02-04,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.60
3,1991-02-05,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.61
4,1991-02-06,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.62
5,1991-02-07,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.63
6,1991-02-08,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.64
7,1991-02-11,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.64
8,1991-02-12,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.64
9,1991-02-13,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.66


In [25]:
tech_funda_fin_filtered["eps_calc"] = tech_funda_fin_filtered["n_income"] / (tech_funda_fin_filtered["total_share_x"] * 10000)
tech_funda_fin_filtered["pe_calc"] = tech_funda_fin_filtered["close_df1"] / tech_funda_fin_filtered["eps_calc"]
tech_funda_fin_filtered.head(50)

C:\Users\HANJ29\AppData\Local\Temp\ipykernel_11188\1541014068.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tech_funda_fin_filtered["eps_calc"] = tech_funda_fin_filtered["n_income"] / (tech_funda_fin_filtered["total_share_x"] * 10000)
C:\Users\HANJ29\AppData\Local\Temp\ipykernel_11188\1541014068.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tech_funda_fin_filtered["pe_calc"] = tech_funda_fin_filtered["close_df1"] / tech_funda_fin_filtered["eps_calc"]


,trade_date,pe,pe_ttm,n_income,eps,basic_eps,diluted_eps,total_revenue,revenue,total_share_x,total_share_y,close_df1,eps_calc,pe_calc
0,1991-01-29,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.60,4.26624,0.375037
1,1991-01-30,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.60,4.26624,0.375037
2,1991-02-04,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.60,4.26624,0.375037
3,1991-02-05,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.61,4.26624,0.377381
4,1991-02-06,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.62,4.26624,0.379725
5,1991-02-07,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.63,4.26624,0.382069
6,1991-02-08,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.64,4.26624,0.384413
7,1991-02-11,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.64,4.26624,0.384413
8,1991-02-12,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.64,4.26624,0.384413
9,1991-02-13,10.7085,10.7085,175968245.3,0.31,0.105,0.105,1.227544e+09,1.227544e+09,4124.668,242955336.0,1.66,4.26624,0.389101


In [27]:
tech_funda_fin_filtered.tail(50)

,trade_date,pe,pe_ttm,n_income,eps,basic_eps,diluted_eps,total_revenue,revenue,total_share_x,total_share_y,close_df1,eps_calc,pe_calc
8028,2024-12-31,7.1215,7.7887,-1.639811e+10,-1.5132,-1.5132,-1.5132,2.198948e+11,2.198948e+11,1.193071e+06,1.193071e+10,7.26,-1.374446,-5.282130
8029,2025-01-02,6.9744,7.7887,-1.639811e+10,-1.5132,-1.5132,-1.5132,2.198948e+11,2.198948e+11,1.193071e+06,1.193071e+10,7.11,-1.374446,-5.172995
8030,2025-01-03,6.8665,7.7887,-1.639811e+10,-1.5132,-1.5132,-1.5132,2.198948e+11,2.198948e+11,1.193071e+06,1.193071e+10,7.00,-1.374446,-5.092963
8031,2025-01-06,6.8469,7.7887,-1.639811e+10,-1.5132,-1.5132,-1.5132,2.198948e+11,2.198948e+11,1.193071e+06,1.193071e+10,6.98,-1.374446,-5.078411
8032,2025-01-07,6.9155,7.7887,-1.639811e+10,-1.5132,-1.5132,-1.5132,2.198948e+11,2.198948e+11,1.193071e+06,1.193071e+10,7.05,-1.374446,-5.129341
8033,2025-01-08,6.8273,7.7887,-1.639811e+10,-1.5132,-1.5132,-1.5132,2.198948e+11,2.198948e+11,1.193071e+06,1.193071e+10,6.96,-1.374446,-5.063860
8034,2025-01-09,6.8174,7.7887,-1.639811e+10,-1.5132,-1.5132,-1.5132,2.198948e+11,2.198948e+11,1.193071e+06,1.193071e+10,6.95,-1.374446,-5.056584
8035,2025-01-10,6.5624,7.7887,-1.639811e+10,-1.5132,-1.5132,-1.5132,2.198948e+11,2.198948e+11,1.193071e+06,1.193071e+10,6.69,-1.374446,-4.867417
8036,2025-01-13,6.6311,7.7887,-1.639811e+10,-1.5132,-1.5132,-1.5132,2.198948e+11,2.198948e+11,1.193071e+06,1.193071e+10,6.76,-1.374446,-4.918347
8037,2025-01-14,6.7782,7.7887,-1.639811e+10,-1.5132,-1.5132,-1.5132,2.198948e+11,2.198948e+11,1.193071e+06,1.193071e+10,6.91,-1.374446,-5.027482


In [16]:
import pandas as pd

# 示例 DataFrame
data = {
    'trade_date': pd.date_range(start='2025-01-01', end='2025-01-31', freq='D'),
    'open': [100 + i for i in range(31)],
    'high': [105 + i for i in range(31)],
    'low': [95 + i for i in range(31)],
    'close': [100 + i for i in range(31)],
}
df = pd.DataFrame(data)

# 将 trade_date 列设置为索引
df.set_index('trade_date', inplace=True)

# 给定的日期
given_date = '2025-01-17'

# 过滤出从给定日期到目前的数据
df_from_given_date = df.loc[given_date:]

# 计算从给定日期到目前的涨跌幅
total_pct_change = (df_from_given_date['close'].iloc[-1] - df_from_given_date['close'].iloc[0]) / df_from_given_date['close'].iloc[0] * 100

# 计算从给定日期到最高价之间的最大涨幅
max_price = df_from_given_date['high'].max()
max_pct_change = (max_price - df_from_given_date['close'].iloc[0]) / df_from_given_date['close'].iloc[0] * 100

# 计算从给定日期到最低价之间的最大跌幅
min_price = df_from_given_date['low'].min()
min_pct_change = (min_price - df_from_given_date['close'].iloc[0]) / df_from_given_date['close'].iloc[0] * 100

# 打印结果
print(f"从 {given_date} 到目前的总涨跌幅: {total_pct_change:.2f}%")
print(f"从 {given_date} 到最高价之间的最大涨幅: {max_pct_change:.2f}%")
print(f"从 {given_date} 到最低价之间的最大跌幅: {min_pct_change:.2f}%")

从 2025-01-17 到目前的总涨跌幅: 12.07%
从 2025-01-17 到最高价之间的最大涨幅: 16.38%
从 2025-01-17 到最低价之间的最大跌幅: -4.31%


In [1]:
import pandas as pd

# 文件路径
root_path_DEV = "C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/"
root_path_PROD = "C:/Users/HANJ29/Applications/btweb/stock_filestore/PROD/"
folder_ds_entry = "dataset_entry/"
# folder_ds_tech_funda = "dataset_tech_funda/"
dataset_training_path = root_path_DEV + "dataset_training/"
dataset_entry_path = root_path_DEV + folder_ds_entry

# 示例 DataFrame
data = {
    'top_or_bottom': [0, 1, 2, 0, 1, 2, 0],
    'top_or_bottom_stat': [1, 1, 2, 0, 0, 2, 1],
    'top_bottom_volatility_stat': [2, 2, 1, 0, 0, 1, 2],
    'top_or_bottom_stat_optimized': [0, 1, 1, 2, 2, 0, 1],
    'top_or_bottom_optimized': [1, 0, 2, 2, 1, 0, 0],
    'top_bottom_volatility_optimized': [2, 1, 0, 0, 1, 2, 2]
}
df = pd.DataFrame(data)

# 统计每列中每个值的数量
result = {}
for column in df.columns:
    result[column] = df[column].value_counts()

# 将统计结果转换为 DataFrame
result_df = pd.DataFrame(result).fillna(0).astype(int)

# 保存统计结果为 CSV 文件
result_df.to_csv(f'{dataset_training_path}value_counts.csv', index_label='value')

print("统计结果已保存为 value_counts.csv")

统计结果已保存为 value_counts.csv


In [3]:
import pandas as pd

# 示例 DataFrame
data = {
    'ts_code': ['000001.SZ'] * 10 + ['000002.SZ'] * 10,
    'trade_date': pd.date_range(start='2020-01-01', periods=10).tolist() + pd.date_range(start='2020-01-01', periods=10).tolist(),
    'feature1': range(20),
    'feature2': range(20, 40),
    'label': [0, 1] * 10
}
df = pd.DataFrame(data)

# 确保 trade_date 是日期类型
df['trade_date'] = pd.to_datetime(df['trade_date'])

# 定义分割函数
def split_train_test(group, train_ratio=0.8):
    # 按 trade_date 排序
    group = group.sort_values('trade_date')
    # 计算分割点
    split_index = int(len(group) * train_ratio)
    # 分割成训练集和测试集
    train = group.iloc[:split_index]
    test = group.iloc[split_index:]
    return train, test

def get_train_test_data(df, group_by="ts_code", train_ratio=0.8):
    # 按 code 分组
    groups = df.groupby(group_by)
    # 对每个分组应用分割函数
    res = [split_train_test(group, train_ratio) for name, group in groups]
    # 拆分训练集和测试集
    train = pd.concat([t[0] for t in res])
    test = pd.concat([t[1] for t in res])
    return train, test

# 合并所有股票的训练集和测试集
train_df,test_df = get_train_test_data(df)

# 打印结果
print("训练集：")
print(train_df)
print("\n测试集：")
print(test_df)

训练集：
      ts_code trade_date  feature1  feature2  label
0   000001.SZ 2020-01-01         0        20      0
1   000001.SZ 2020-01-02         1        21      1
2   000001.SZ 2020-01-03         2        22      0
3   000001.SZ 2020-01-04         3        23      1
4   000001.SZ 2020-01-05         4        24      0
5   000001.SZ 2020-01-06         5        25      1
6   000001.SZ 2020-01-07         6        26      0
7   000001.SZ 2020-01-08         7        27      1
10  000002.SZ 2020-01-01        10        30      0
11  000002.SZ 2020-01-02        11        31      1
12  000002.SZ 2020-01-03        12        32      0
13  000002.SZ 2020-01-04        13        33      1
14  000002.SZ 2020-01-05        14        34      0
15  000002.SZ 2020-01-06        15        35      1
16  000002.SZ 2020-01-07        16        36      0
17  000002.SZ 2020-01-08        17        37      1

测试集：
      ts_code trade_date  feature1  feature2  label
8   000001.SZ 2020-01-09         8        28      0
9